# Анализ событий с SQL и pandas

Учебный проект. Исследуем просмотры и события выполнения задач, строим витрину SQLite и сравниваем временные показатели. Все выводы рассчитаны на синтетических данных.

In [1]:
import pandas as pd

from demo_data import connect_demo

conn = connect_demo()

## Просмотр данных

In [2]:
pageviews_df = pd.read_sql('SELECT * FROM pageviews;', conn)
pageviews_df.head()

,uid,datetime
0,user_01,2025-01-12 19:00:00
1,user_01,2025-01-12 20:00:00
2,user_01,2025-01-12 21:00:00
3,user_01,2025-01-12 22:00:00
4,user_02,2025-01-12 20:00:00


In [3]:
pageviews_schema = pd.read_sql('PRAGMA table_info(pageviews);', conn)
print(pageviews_schema)

   cid      name  type  notnull dflt_value  pk
0    0       uid  TEXT        1       None   0
1    1  datetime  TEXT        1       None   0


In [4]:
pageviews_preview = pd.read_sql('SELECT * FROM pageviews LIMIT 10;', conn)
pageviews_preview

,uid,datetime
0,user_01,2025-01-12 19:00:00
1,user_01,2025-01-12 20:00:00
2,user_01,2025-01-12 21:00:00
3,user_01,2025-01-12 22:00:00
4,user_02,2025-01-12 20:00:00
5,user_02,2025-01-12 21:00:00
6,user_02,2025-01-12 22:00:00
7,user_02,2025-01-12 23:00:00
8,user_02,2025-01-13 00:00:00
9,user_02,2025-01-13 01:00:00


In [5]:
query = '''
SELECT uid, datetime
FROM pageviews
WHERE uid GLOB 'user_*'
ORDER BY uid ASC;
'''
pageviews = pd.read_sql(query, conn)
pageviews = pageviews.set_index('datetime')
pageviews.index = pd.to_datetime(pageviews.index)
pageviews.head()

,uid
datetime,
2025-01-12 19:00:00,user_01
2025-01-12 20:00:00,user_01
2025-01-12 21:00:00,user_01
2025-01-12 22:00:00,user_01
2025-01-12 20:00:00,user_02


In [6]:
pageviews.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 47 entries, 2025-01-12 19:00:00 to 2025-01-13 01:00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   uid     47 non-null     object
dtypes: object(1)
memory usage: 752.0+ bytes


## Отбор событий через подзапрос

In [7]:
checker_schema = pd.read_sql('PRAGMA table_info(checker);', conn)
print(checker_schema)

   cid       name     type  notnull dflt_value  pk
0    0        uid     TEXT        1       None   0
1    1    labname     TEXT        1       None   0
2    2  timestamp     TEXT        1       None   0
3    3     status     TEXT        1       None   0
4    4  numTrials  INTEGER        1       None   0


In [8]:
checker_preview = pd.read_sql('SELECT * FROM checker LIMIT 10;', conn)
checker_preview

,uid,labname,timestamp,status,numTrials
0,user_01,task_04,2025-01-10 14:00:00,ready,1
1,user_01,task_04b,2025-01-11 13:00:00,ready,1
2,user_01,task_05,2025-01-12 12:00:00,ready,1
3,user_01,task_06,2025-01-13 11:00:00,ready,1
4,user_01,task_06b,2025-01-14 10:00:00,ready,1
5,user_01,project,2025-01-15 09:00:00,ready,1
6,user_01,task_04,2025-01-10 18:00:00,ready,2
7,user_02,task_04,2025-01-10 13:00:00,ready,1
8,user_02,task_04b,2025-01-11 11:00:00,ready,1
9,user_02,task_05,2025-01-12 09:00:00,ready,1


In [9]:
query = """
SELECT COUNT(*) AS cnt
FROM pageviews
WHERE uid IN (
    SELECT uid
    FROM checker
    WHERE status = 'ready'
        AND numTrials = 1
        AND labname IN (
            'task_04',
            'task_04b',
            'task_05',
            'task_06',
            'task_06b',
            'project'
        )
);
"""
checkers = pd.read_sql(query, conn)
checkers

,cnt
0,47


## Витрина по пользователям и задачам

In [10]:
query = """
SELECT
    ch.uid,
    ch.labname,
    MIN(ch.timestamp) AS first_commit_ts,
    MIN(pv.datetime) AS first_view_ts
FROM checker ch
LEFT JOIN pageviews pv
    ON ch.uid = pv.uid
WHERE ch.status = 'ready'
    AND ch.numTrials = 1
    AND ch.labname IN (
        'task_04',
        'task_04b',
        'task_05',
        'task_06',
        'task_06b',
        'project'
    )
    AND ch.uid GLOB 'user_*'
GROUP BY
    ch.uid,
    ch.labname;
"""
datamart = pd.read_sql(
    query,
    conn,
    parse_dates=['first_commit_ts', 'first_view_ts'],
)
datamart.to_sql(
    'datamart',
    conn,
    if_exists='replace',
    index=False,
)
datamart.head()

,uid,labname,first_commit_ts,first_view_ts
0,user_01,project,2025-01-15 09:00:00,2025-01-12 19:00:00
1,user_01,task_04,2025-01-10 14:00:00,2025-01-12 19:00:00
2,user_01,task_04b,2025-01-11 13:00:00,2025-01-12 19:00:00
3,user_01,task_05,2025-01-12 12:00:00,2025-01-12 19:00:00
4,user_01,task_06,2025-01-13 11:00:00,2025-01-12 19:00:00


In [11]:
datamart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              96 non-null     object        
 1   labname          96 non-null     object        
 2   first_commit_ts  96 non-null     datetime64[ns]
 3   first_view_ts    60 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.1+ KB


In [12]:
test = datamart[datamart['first_view_ts'].notna()].copy()
test.head()

,uid,labname,first_commit_ts,first_view_ts
0,user_01,project,2025-01-15 09:00:00,2025-01-12 19:00:00
1,user_01,task_04,2025-01-10 14:00:00,2025-01-12 19:00:00
2,user_01,task_04b,2025-01-11 13:00:00,2025-01-12 19:00:00
3,user_01,task_05,2025-01-12 12:00:00,2025-01-12 19:00:00
4,user_01,task_06,2025-01-13 11:00:00,2025-01-12 19:00:00


In [13]:
control = datamart[datamart['first_view_ts'].isna()].copy()
control.head()

,uid,labname,first_commit_ts,first_view_ts
60,user_11,project,2025-01-14 19:00:00,NaT
61,user_11,task_04,2025-01-10 04:00:00,NaT
62,user_11,task_04b,2025-01-10 17:00:00,NaT
63,user_11,task_05,2025-01-12 05:00:00,NaT
64,user_11,task_06,2025-01-12 18:00:00,NaT


In [14]:
control['first_view_ts'] = control['first_view_ts'].fillna(
    test['first_view_ts'].mean()
)
control.head()

,uid,labname,first_commit_ts,first_view_ts
60,user_11,project,2025-01-14 19:00:00,2025-01-12 19:00:00.000000512
61,user_11,task_04,2025-01-10 04:00:00,2025-01-12 19:00:00.000000512
62,user_11,task_04b,2025-01-10 17:00:00,2025-01-12 19:00:00.000000512
63,user_11,task_05,2025-01-12 05:00:00,2025-01-12 19:00:00.000000512
64,user_11,task_06,2025-01-12 18:00:00,2025-01-12 19:00:00.000000512


In [15]:
test.to_sql(
    'test',
    conn,
    if_exists='replace',
    index=False,
)
control.to_sql(
    'control',
    conn,
    if_exists='replace',
    index=False,
)

36

Группы сформированы по наличию просмотра. Для группы без просмотров условная точка отсчёта — среднее время первого просмотра в наблюдаемой группе.

## Агрегации и корреляция

In [16]:
print(pd.read_sql('PRAGMA table_info(test);', conn))

   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


In [17]:
pd.read_sql('SELECT * FROM test LIMIT 10;', conn)

,uid,labname,first_commit_ts,first_view_ts
0,user_01,project,2025-01-15 09:00:00,2025-01-12 19:00:00
1,user_01,task_04,2025-01-10 14:00:00,2025-01-12 19:00:00
2,user_01,task_04b,2025-01-11 13:00:00,2025-01-12 19:00:00
3,user_01,task_05,2025-01-12 12:00:00,2025-01-12 19:00:00
4,user_01,task_06,2025-01-13 11:00:00,2025-01-12 19:00:00
5,user_01,task_06b,2025-01-14 10:00:00,2025-01-12 19:00:00
6,user_02,project,2025-01-15 03:00:00,2025-01-12 20:00:00
7,user_02,task_04,2025-01-10 13:00:00,2025-01-12 20:00:00
8,user_02,task_04b,2025-01-11 11:00:00,2025-01-12 20:00:00
9,user_02,task_05,2025-01-12 09:00:00,2025-01-12 20:00:00


In [18]:
pd.read_sql('SELECT * FROM deadlines;', conn)  ## Смотрим структуру таблицы deadlines

,labs,deadlines
0,task_04,1736532000
1,task_04b,1736618400
2,task_05,1736704800
3,task_06,1736791200
4,task_06b,1736877600
5,project,1736964000


In [19]:
query = """
SELECT
    t.uid,
    MIN(
        (
            julianday(t.first_commit_ts)
            - julianday(datetime(d.deadlines, 'unixepoch'))
        ) * 24
    ) AS df_min
FROM test t
LEFT JOIN deadlines d
    ON t.labname = d.labs
WHERE t.labname IN (
    'task_04',
    'task_04b',
    'task_05',
    'task_06',
    'task_06b'
)
GROUP BY
    t.uid;
"""
df_min = pd.read_sql(query, conn)
df_min

,uid,df_min
0,user_01,-8.0
1,user_02,-13.0
2,user_03,-18.0
3,user_04,-23.0
4,user_05,-23.0
5,user_06,-21.0
6,user_07,-24.0
7,user_08,-20.0
8,user_09,-25.0
9,user_10,-23.0


In [20]:
query = """
SELECT
    t.uid,
    MAX(
        (
            julianday(t.first_commit_ts)
            - julianday(datetime(d.deadlines, 'unixepoch'))
        ) * 24
    ) AS df_max
FROM test t
LEFT JOIN deadlines d
    ON t.labname = d.labs
WHERE t.labname IN (
    'task_04',
    'task_04b',
    'task_05',
    'task_06',
    'task_06b'
)
GROUP BY
    t.uid;
"""
df_max = pd.read_sql(query, conn)
df_max

,uid,df_max
0,user_01,-4.0
1,user_02,-5.0
2,user_03,-6.0
3,user_04,-7.0
4,user_05,-5.0
5,user_06,-4.0
6,user_07,-8.0
7,user_08,-4.0
8,user_09,-7.0
9,user_10,-7.0


In [21]:
query = """
SELECT
    AVG(
        (
            julianday(t.first_commit_ts)
            - julianday(datetime(d.deadlines, 'unixepoch'))
        ) * 24
    ) AS df_avg
FROM test t
LEFT JOIN deadlines d
    ON t.labname = d.labs
WHERE t.labname IN (
    'task_04',
    'task_04b',
    'task_05',
    'task_06',
    'task_06b'
)
"""
df_avg = pd.read_sql(query, conn)
df_avg.head()

,df_avg
0,-12.6


In [22]:
query = """
SELECT
    t.uid,
    AVG(
        (
            julianday(t.first_commit_ts)
            - julianday(datetime(dl.deadlines, 'unixepoch'))
        ) * 24
    ) AS avg_diff,
    COALESCE(MAX(pv.pageviews), 0) AS pageviews
FROM test t
LEFT JOIN deadlines dl
    ON t.labname = dl.labs
LEFT JOIN (
    SELECT uid, COUNT(*) AS pageviews
    FROM pageviews
    GROUP BY uid
) pv ON t.uid = pv.uid
WHERE t.labname IN (
    'task_04',
    'task_04b',
    'task_05',
    'task_06',
    'task_06b'
)
GROUP BY
    t.uid;
"""
views_diff = pd.read_sql(query, conn)
views_diff.head()

,uid,avg_diff,pageviews
0,user_01,-6.0,4
1,user_02,-9.0,7
2,user_03,-12.0,2
3,user_04,-15.0,5
4,user_05,-13.4,8


In [23]:
pearson_r = views_diff['pageviews'].corr(views_diff['avg_diff'])
print(f'Корреляция Пирсона: {pearson_r:.3f}')

Корреляция Пирсона: -0.084


Количество просмотров агрегируется до соединения с витриной, чтобы одна сессия не считалась повторно за каждую задачу.

## Сравнение показателей до и после просмотра

In [24]:
query = """
SELECT
    x.time,
    AVG(x.diff_hours) AS avg_diff
FROM (
    SELECT
        t.uid,
        t.labname,
        CASE
            WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
            ELSE 'after'
        END AS time,
        (
            julianday(t.first_commit_ts)
            - julianday(datetime(d.deadlines, 'unixepoch'))
        ) * 24 AS diff_hours
    FROM test t
    JOIN deadlines d
        ON t.labname = d.labs
    WHERE t.labname <> 'project'
) x
WHERE x.uid IN (
    SELECT
        t.uid
    FROM test t
    WHERE t.labname <> 'project'
    GROUP BY t.uid
    HAVING COUNT(
        DISTINCT CASE
            WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
            ELSE 'after'
        END
    ) = 2
)
GROUP BY x.time;
"""
test_results = pd.read_sql(query, conn)
test_results.head()

,time,avg_diff
0,after,-13.473684
1,before,-12.064516


In [25]:
query = """
SELECT
    x.time,
    AVG(x.diff_hours) AS avg_diff
FROM (
    SELECT
        c.uid,
        c.labname,
        CASE
            WHEN c.first_commit_ts < c.first_view_ts THEN 'before'
            ELSE 'after'
        END AS time,
        (
            julianday(c.first_commit_ts)
            - julianday(datetime(d.deadlines, 'unixepoch'))
        ) * 24 AS diff_hours
    FROM control c
    JOIN deadlines d
        ON c.labname = d.labs
    WHERE c.labname <> 'project'
) x
WHERE x.uid IN (
    SELECT
        c.uid
    FROM control c
    WHERE c.labname <> 'project'
    GROUP BY c.uid
    HAVING COUNT(
        DISTINCT CASE
            WHEN c.first_commit_ts < c.first_view_ts THEN 'before'
            ELSE 'after'
        END
    ) = 2
)
GROUP BY x.time;
"""
control_results = pd.read_sql(query, conn)
control_results.head()

,time,avg_diff
0,after,-13.000000
1,before,-15.157895


Сравнение описывает различия между группами и периодами на синтетических данных. Для оценки причинного эффекта потребовался бы эксперимент с рандомизацией.

In [26]:
conn.close()